In [85]:
import ollama
import json
from IPython.display import Markdown, display, update_display

from scraper import fetch_website_links, fetch_website_contents
url = "https://edwarddonner.com"


In [68]:
contents = fetch_website_links(url)
print(contents)

['https://edwarddonner.com/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2025/11/11/ai-live-event/', 'https://edwarddonner.com/2025/11/11/ai-l

In [69]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [70]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [71]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [72]:
def select_relevant_links(url):
    response = ollama.chat(
        model = "llama3.2",
        format = "json",
        messages = [
            {'role':'system', 'content' : link_system_prompt},
            {'role':'user', 'content': get_links_user_prompt(url)}],
           
    )
    return json.loads(response.message.content)

In [73]:
links = select_relevant_links("https://edwarddonner.com")
print(links)

{'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}, {'type': 'company page', 'url': 'https://edwarddonner.com/'}, {'type': 'Careers/Jobs page', 'url': 'https://www.linkedin.com/in/eddonner/'}]}


In [74]:
links = select_relevant_links("https://en.wikipedia.org/wiki/Ashok_Agrawala")
print(links)

print(type(links))

{'links': [{'type': 'About page', 'url': 'https://en.wikipedia.org/wiki/Ashok_Agrawala'}, {'type': 'Company/University page', 'url': 'https://www.cs.umd.edu/ashok50'}, {'type': 'Research pages', 'url': 'https://web.archive.org/web/20150205062907/http://www.mindlab.umd.edu/publications.html'}, {'type': 'Social media and online presence', 'url': 'http://weblogs.baltimoresun.com/news/technology/2011/08/cool_app_university_app_turns.html'}, {'type': 'Biography and achievements', 'url': 'https://dblp.org/pid/a/AshokKAgrawala'}, {'type': 'Awards and recognition', 'url': 'https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Universal_Code_of_Conduct'}, {'type': 'Professional affiliations', 'url': 'https://en.wikipedia.org/w/index.php?title=Category:IEEE_fellows'}, {'type': 'Fellow of the AAAS', 'url': 'https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use'}]}
<class 'dict'>


In [75]:
def get_page_content_and_links(url):
    content = fetch_website_contents(url)
    relevant_urls = select_relevant_links(url)
    result = f"##Welcome to the page data###\n\n\n{content}\n\n##URLS\n\n"
    for link in relevant_urls['links']:
        result+= f"Type:{link['type']}\n"
        result+= f"URL:{link['url']}\n"

    return result

In [76]:
page_info = get_page_content_and_links("https://edwarddonner.com")

In [77]:
print(page_info)

##Welcome to the page data###


Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, 

In [78]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [79]:
def brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += get_page_content_and_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [80]:
 print(brochure_user_prompt("Edward Donner", "https://edwarddonner.com"))


You are looking at a company called: Edward Donner
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


##Welcome to the page data###


Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on f

In [87]:
def create_brochure(company_name, url):
    response = ollama.chat(
        model = "llama3.2",
        messages = [{
            'role':'system','content':brochure_system_prompt,
            'role':'user', 'content':brochure_user_prompt(company_name,url)
        }]
    )
    return display(Markdown(response.message.content))

In [88]:
print(create_brochure("Edward Donner", "https://edwarddonner.com"))

# Edward Donner
## About Us

Welcome to Edward Donner, a company passionate about applying AI to make a positive impact on people's lives. Our mission is to help individuals discover their potential and pursue their reason for being.

## Our Story

We're led by Ed, a co-founder and CTO of Nebula.io, where we're using AI to create innovative solutions. Previously, Ed was the founder and CEO of AI startup untapt, acquired in 2021. With over 400,000 students enrolled across 190 countries, our Udemy courses have become best-selling, top-rated resources for learning about LLMs.

## What We Do

* Create engaging AI training content through our Udemy courses
* Develop cutting-edge AI solutions through Nebula.io
* Host live events and arenas where AI meets diplomacy and strategy (like Connect Four: Outsmart)

## Stay in Touch

Want to stay up-to-date on our latest projects, news, and insights? Subscribe to our newsletter by entering your email address below.

[Subscribe to Newsletter]

Follow us on social media to explore more about our journey:

* LinkedIn
* Twitter
* Facebook

None
